In [1]:
import sys
import torch
import transformers
import pandas as pd

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print(f"Device: {'cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')}")

/Users/shaheeraslam/miniforge3/envs/base-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:52:34) [Clang 18.1.8 ]
PyTorch: 2.11.0
Transformers: 5.9.0
CUDA available: False
MPS available: True
Device: mps


#### Cell 2 — load training_dataset.xlsx, encode labels

In [2]:
df = pd.read_excel("../training_dataset.xlsx")
print(f"Loaded: {df.shape}")
print(df["Label"].value_counts())

labels = sorted(df["Label"].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"\nLabel mapping: {label2id}")

df["label_id"] = df["Label"].map(label2id)
print(f"\nNulls in label_id (should be 0): {df['label_id'].isna().sum()}")
print(df[["Label", "label_id"]].drop_duplicates().sort_values("label_id"))

Loaded: (592, 5)
Label
UNSAFE_EXECUTION    208
LOOP                146
SUCCESS             129
HALLUCINATION       109
Name: count, dtype: int64

Label mapping: {'HALLUCINATION': 0, 'LOOP': 1, 'SUCCESS': 2, 'UNSAFE_EXECUTION': 3}

Nulls in label_id (should be 0): 0
                Label  label_id
354     HALLUCINATION         0
0                LOOP         1
356           SUCCESS         2
146  UNSAFE_EXECUTION         3


#### Cell 3 — grouped + stratified train/val/test split (keyed on Parent Trace ID)

In [3]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# Step 1: carve off a held-out test set (~20%), grouped by Parent Trace ID so no
# trace and its truncated/synthetic sibling ever land on opposite sides of a split
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_val_idx, test_idx = next(sgkf_test.split(df, df["label_id"], groups=df["Parent Trace ID"]))

test_df = df.iloc[test_idx].reset_index(drop=True)
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)

# Step 2: split the remaining 80% into train/val (~64%/16% of total), same grouped approach
sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_val.split(train_val_df, train_val_df["label_id"], groups=train_val_df["Parent Trace ID"]))

val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} ({100*len(train_df)/len(df):.1f}%)")
print(f"Val:   {len(val_df)} ({100*len(val_df)/len(df):.1f}%)")
print(f"Test:  {len(test_df)} ({100*len(test_df)/len(df):.1f}%)")

print("\nLabel distribution per split:")
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print(split["Label"].value_counts(normalize=True).round(3))

# critical check: no Parent Trace ID group should span across splits
train_groups = set(train_df["Parent Trace ID"])
val_groups = set(val_df["Parent Trace ID"])
test_groups = set(test_df["Parent Trace ID"])
print("\nGroup leakage check (all should be 0):")
print("train/val overlap:", len(train_groups & val_groups))
print("train/test overlap:", len(train_groups & test_groups))
print("val/test overlap:", len(val_groups & test_groups))

Train: 410 (69.3%)
Val:   99 (16.7%)
Test:  83 (14.0%)

Label distribution per split:

Train:
Label
UNSAFE_EXECUTION    0.395
LOOP                0.222
SUCCESS             0.198
HALLUCINATION       0.185
Name: proportion, dtype: float64

Val:
Label
LOOP                0.374
SUCCESS             0.242
UNSAFE_EXECUTION    0.212
HALLUCINATION       0.172
Name: proportion, dtype: float64

Test:
Label
UNSAFE_EXECUTION    0.301
SUCCESS             0.289
LOOP                0.217
HALLUCINATION       0.193
Name: proportion, dtype: float64

Group leakage check (all should be 0):
train/val overlap: 0
train/test overlap: 0
val/test overlap: 0


#### Cell 4 — check Parent Trace ID group sizes before deciding how to handle the split imbalance

In [4]:
group_sizes = df.groupby("Parent Trace ID").size().sort_values(ascending=False)
print("Largest groups (by Parent Trace ID):")
print(group_sizes.head(10))
print()
print("Group size distribution:")
print(group_sizes.describe())
print()
print("Number of groups with >5 rows:", (group_sizes > 5).sum())
print("Total rows sitting in groups >5:", group_sizes[group_sizes > 5].sum())

Largest groups (by Parent Trace ID):
Parent Trace ID
42c741a6,48b72100,759addd3,9bd80c16,a08a1ca5,aa5d9d13,c4815d5c,fe367381,0d896eeb,c694b995    40
19edaca4,23046d46,4484b485                                                                   28
aa5d9d13                                                                                      9
c1762328,83add5f7,a02f0cb8                                                                    7
4b92af45,d9c46fce,46b63241                                                                    7
0d896eeb,9bd80c16,48b72100                                                                    7
a02f0cb8,17804dec,423d4398                                                                    6
4b92af45,de469b78,36fdff20                                                                    6
c694b995,c4815d5c,759addd3                                                                    6
0efc9456,423d4398,17804dec                                                         

#### Cell 5 — tokenizer setup

In [10]:

from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_lengths = df["Trace Content"].apply(lambda t: len(tokenizer.encode(t)))
print(sample_lengths.describe())
print(f"\n% of traces over 512 tokens: {(sample_lengths > 512).mean()*100:.1f}%")

[transformers] Could not extract SentencePiece model from /Users/shaheeraslam/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

In [8]:
import sys
print(sys.executable)

try:
    import sentencepiece
    print("sentencepiece OK:", sentencepiece.__file__)
except ImportError as e:
    print("sentencepiece FAILED:", e)

/Users/shaheeraslam/miniforge3/envs/base-ml/bin/python
sentencepiece OK: /Users/shaheeraslam/miniforge3/envs/base-ml/lib/python3.11/site-packages/sentencepiece/__init__.py


In [9]:
import shutil
from pathlib import Path

cache_path = Path.home() / ".cache" / "huggingface" / "hub" / "models--microsoft--deberta-v3-base"
if cache_path.exists():
    shutil.rmtree(cache_path)
    print(f"Cleared: {cache_path}")
else:
    print("Nothing cached at that path")

Cleared: /Users/shaheeraslam/.cache/huggingface/hub/models--microsoft--deberta-v3-base
